# 04. LightGBM Model Training — Baseline (no preprocessing)

This notebook trains a **LightGBM** classifier with **default settings** on all **103 signal features**, without any preprocessing, to get a first baseline score.

Why no preprocessing is acceptable here:
- **Scaling** is not needed: tree models split on value thresholds, so feature scale does not matter.
- **Imputation** is not needed: the processed data has no missing values (see `data/processed/README.md`).
- **Feature selection** is not done: this is the *full feature set* baseline.

Rules followed:
- Train on `train.csv` only, score on `val.csv` only.
- `test.csv` is **not loaded** in this notebook.
- `record_id`, `patient_id` and `figo_label` are not used as input features.
- Positive class = abnormal (`figo_label = 1`). Threshold = 0.5 (default).

## Step 1: Setup and imports

In [1]:
import sys
sys.path.append('../..')

import time
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score, confusion_matrix

from config import config
from src.data_preprocessing import get_feature_columns

## Step 2: Load the training and validation sets

The test set stays untouched until the final evaluation.

In [2]:
# Load train and validation sets only (test set stays untouched)
train_df = pd.read_csv(config.PROCESSED_TRAIN_FILE)
val_df = pd.read_csv(config.PROCESSED_VAL_FILE)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Train shape: (14540, 106)
Validation shape: (3115, 106)


## Step 3: Separate features (X) and target (y)

In [3]:
# Get the 103 signal features (IDs and target are excluded)
feature_cols = get_feature_columns(train_df)

X_train = train_df[feature_cols]
y_train = train_df[config.TARGET_COLUMN]

X_val = val_df[feature_cols]
y_val = val_df[config.TARGET_COLUMN]

print("Train:", X_train.shape, " Validation:", X_val.shape)

Number of feature columns found: 103
Train: (14540, 103)  Validation: (3115, 103)


## Step 4: Create and train the LightGBM model

Default hyperparameters; only the random seed is fixed for reproducibility.

In [4]:
# Create LightGBM with default settings (only the seed is fixed)
model = LGBMClassifier(random_state=config.RANDOM_STATE)

# Train on the training set only
start_time = time.time()
model.fit(X_train, y_train)
train_seconds = time.time() - start_time

print("Training time (seconds):", round(train_seconds, 2))

[LightGBM] [Info] Number of positive: 7821, number of negative: 6719
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008580 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 22881
[LightGBM] [Info] Number of data points in the train set: 14540, number of used features: 103
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.537895 -> initscore=0.151873
[LightGBM] [Info] Start training from score 0.151873
Training time (seconds): 0.86


## Step 5: Predict on the validation set

In [5]:
# Class predictions (threshold 0.5 by default)
val_pred = model.predict(X_val)

# Probability of abnormal (1), needed for ROC-AUC
val_prob = model.predict_proba(X_val)[:, 1]

## Step 6: Validation metrics

Positive class = abnormal (`figo_label = 1`).

In [6]:
print("Accuracy:         ", round(accuracy_score(y_val, val_pred), 4))
print("Precision:        ", round(precision_score(y_val, val_pred), 4))
print("Recall:           ", round(recall_score(y_val, val_pred), 4))
print("F1-score:         ", round(f1_score(y_val, val_pred), 4))
print("ROC-AUC:          ", round(roc_auc_score(y_val, val_prob), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_val, val_pred), 4))

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_val, val_pred))

Accuracy:          0.9297
Precision:         0.9344
Recall:            0.935
F1-score:          0.9347
ROC-AUC:           0.9844
Balanced accuracy: 0.9293
Confusion matrix [[TN, FP], [FN, TP]]:
[[1329  110]
 [ 109 1567]]


## Step 7: Check for overfitting

If the training F1 is much higher than the validation F1, the model is overfitting.

In [7]:
# Training F1, compared with validation F1 from Step 6
train_pred = model.predict(X_train)
print("Training F1:  ", round(f1_score(y_train, train_pred), 4))
print("Validation F1:", round(f1_score(y_val, val_pred), 4))

Training F1:   0.994
Validation F1: 0.9347
